# Implementing `Attention Is All You Need Vaswani et al. 2017` From Scratch 

## Part-1: A Foundational Analysis of Attention Is All You Need

### Section-1: From Sequential Processing to Transformers

#### 1.1: The Drawbacks of Recurrence

Before 2017 when **Attention Is All You Need** introduced the Transformer the [*Recurrent Neural Networks (RNNs)*](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-recurrent-neural-networks)  or more like their sophisticated variants like [*Long Short-Term Memory (LSTM)*](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) and [*Gated Recurrent Unit (GRU)*](https://medium.com/@anishnama20/understanding-gated-recurrent-unit-gru-in-deep-learning-2e54923f3e2) networks were firmly established as the state-of-the-art for tasks such as machine translation. The basic principle of of these models are inherently sequential.<br>
Starting from basics they process a sequence token by token generating a hidden state $h_t$ that is a function of the previous hidden state $h_{t-1}$ and the input for the current position $t.$ <br>
This idea while just shows basic common sense contains a critical flaw in the context of modern hardware that its ***inherently sequential nature prevents parallelisation within training examples.*** The computation of the hidden state for step $t$ is dependent on the completion of the computation for step $t-1.$ This flaw creates a dependency chain that cannot be broken i.e. forcing the model to process information one step at a time. This stands in direct opposition to the architecture of modern processors like *Graphics Processing Units (GPUs)* which derive their immense power from performing thousands of operations in parallel.<br>

Therefore, this fundamental mismatch between the sequential software and parallel hardware created a computational bottleneck that severely limited training speed and the ability to effectively train on longer sequences also where memory constraints also restrict batching across examples. The problem was not just that RNNs were slow but also they were architecturally incapable of fully utilising the computational power that had become available hence motivating the search for a non-sequential alternative.

#### 1.2 The Limitations of Convolutional Approaches

For overcoming sequential bottleneck of RNNs some researchers turned to Convolutional Neural Networks (CNNs). Models such as ByteNet and ConvS2S utilised convolutional layers to compute hidden representations for all input and output positions in parallel successfully addressing the parallelisation issue. <br>

But this approach introduced a different kind of limitation: a "path length" problem. CNNs operate through fixed-size kernels means a single layer can only model relationships between tokens that fall within its local field. To capture a dependency between two distant words in a sentence, for example, the first and last words a signal must traverse a stack of convolutional layers. Therefore number of operations and length of the signal path required to relate two arbitrary positions grows with the distance between them linearly in the case of ConvS2S and logarithmically for ByteNet.<br>
Longer paths are known to complicate the flow of gradients during training making it **more difficult to learn dependencies between distant positions.** While CNNs solved the parallelisation problem of RNNs they also introduced a strong structural bias towards local interactions which was a significant flaw for complex language understanding tasks that often rely on long-range context. Hence the ideal model needed to solve both the parallelization and the path length problems simultaneously. 

#### 1.3 The Research Question and Proposed Solution

The *transformer's* authors stated goal was to ***propose a new simple network architecture the Transformer, based solely on attention mechanisms dispensing with recurrence and convolutions entirely.*** The core hypothesis was that the attention mechanism which was previously used as an enhancement in conjunction with RNNs was powerful enough to stand on its own.<br>
This will lead us to central research question of the paper: *can a model based just on attention without any sequential recurrence or local convolutions, effectively model complex sequence-to-sequence tasks and in doing so can overcome the parallelization and path length limitations of prior architectures?*

> Hence the title **Attention Is All You Need** tells us that attention is not just a useful component but the only component necessary for building powerful sequence models.

### Section-2: Understanding the Transformer Architecture

The Transformer's core lies in its simplicity where internal workings of its components represent a fundamental departure from previous models. Each piece is carefully designed to contribute to a system that is both highly expressive and exceptionally efficient to train.<br>

#### 2.1 The High-Level Encoder-Decoder Framework

At high level the Transformer follows the classic encoder-decoder structure that is standard for [sequence transduction problems](https://machinelearningmastery.com/transduction-in-machine-learning/). 
The encoder's role is to map an input sequence of symbol representations $x_1,...,x_n$ into a sequence of continuous representations $z = z_1,...,z_n.$ Given this representation $z$ the decoder then generates an output sequence $y_1,...,y_m$ one symbol at a time. <br>
This generation process is auto-regressive means that at each step the model consumes the previously generated symbols as additional input to produce the next one.
Both the encoder and decoder are composed of a stack of N=6 identical layers. This design is a key principle allowing the model to build progressively more abstract and complex representations of the sequence at each level. The true innovation, however, lies not in this overall framework but in the composition of the layers themselves which are built not from recurrent cells but from self-attention and simple feed-forward networks.

#### 2.2 The Core of the Transformer: Scaled Dot-Product Attention

The core mechanism powering the Transformer is a specific form of attention called **Scaled Dot-Product Attention.**<br>
An attention function is described as mapping a query and a set of key-value pairs to an output. The output is a weighted sum of the values, where the weight assigned to each value is determined by a compatibility function between the query and the corresponding key.

The specific formula for Scaled Dot-Product Attention is:<br>
$[
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
]$

where $Q$, $K$ and $V$ are matrices packing together the queries, keys, and values.

For example consider translating the sentence: ***The cat sat on the mat.***<br>
When the model is processing the word "sat" it needs to understand the context specially who sat. Hence in our scenario:

1. The Query (Q) is a vector representing the current word of focus: "sat."
2. The model compares this query against a set of **Keys (K)** which are vectors representing all words in the sentence ("The," "cat," "sat," etc.). The dot product $QK^T$ calculates a compatibility score between "sat" and every other word. A high score for the key associated with "cat" indicates high relevance.
3. These raw scores are scaled by dividing by $\frac{1}{\sqrt{d_k}}$ where $d_k$ is the dimension of the key vectors.
4. A softmax function is applied to these scaled scores converting them into a set of weights that sum to 1.  
   These weights represent the amount of "attention" the word "sat" should pay to every other word. The weight for "cat" would be high.
5. Finally these weights are used to compute a weighted average of the **Values (V)** another set of vectors representing each word. The result is a new contextually-aware representation for "sat" that is heavily informed by the information from "cat."


The scaling factor, $\frac{1}{\sqrt{d_k}}$ is not a minor hyperparameter but it is a critical component for stabilising the learning process. The authors write that for large values of $d_k$ the magnitude of the dot products can become very large.<br>
This pushes the softmax function into regions where its gradients are extremely small which is a phenomenon known as **saturation**. When gradients vanish the model receives almost no signal to update its weights and learning effectively stalls.  

The authors provide a statistical justification: if the components of the query and key vectors are independent random variables with a mean of 0 and variance of 1, their dot product has a mean of 0 but a variance of $d_k.$<br>
By dividing the dot products by  $\sqrt{d_k}$ (the standard deviation) the variance is re-normalized to 1 keeping the inputs to the softmax function in a well-behaved range and ensuring stable gradients throughout training.






## Part-2: Implementation from Scratch of Attention Is All You Need

In [ ]:
import math
import copy
import time
import torch
import numpy as np
import torch.nn as nn
import seaborn as sns
import torch.optim as optim
import torch.utils.data as data
import matplotlib.pyplot as plt
from torchtext.data.metrics import bleu_score
from torch.utils.data import DataLoader, random_split